In [1]:
import gc
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
import s3_utils

In [2]:
FOOTNOTE_TEXT = "* Note: 0.2, 1.0, ... are percentiles of top model chunks (MLP layers or attention heads) selected by calculated impact or output probability."

# Map models to their active parameter percentage from JSON output
PARAM_PERCENTAGES = {
    "Original": 100.0,
    "Fine-Tuned (Attn 0.2)": 0.0751,
    "Fine-Tuned (Attn 0.5)": 0.1501,
    "Fine-Tuned (Attn 0.8)": 0.2502,
    "Fine-Tuned (Attn 1)": 0.3003,
    "Fine-Tuned (Attn 5)": 1.5013,
    "Fine-Tuned (Attn 10)": 3.0026,
    "Fine-Tuned (Attn 20)": 6.0051,
    "Fine-Tuned (Attn Impact MLP 0.2)": 3.828,
    "Fine-Tuned (Attn Impact MLP 0.5)": 7.656,
    "Fine-Tuned (Attn Impact MLP 0.8)": 11.509,
    "Fine-Tuned (Attn Impact MLP 1)": 12.81,
    "Fine-Tuned (Attn Impact MLP 2)": 21.8671,
    "Fine-Tuned (Attn Impact MLP 5)": 45.2854,
    "Fine-Tuned (Attn Impact MLP 10)": 56.7945,
    "Fine-Tuned (Attn Impact MLP 20)": 63.55,
    "Fine-Tuned (Full)": 100,
}

In [ ]:
def format_model_name(name: str) -> str:
    """Appends the exact active parameter percentage to the model name."""
    pct = PARAM_PERCENTAGES.get(name)
    if pct is None:
        return name
    if pct == 100.0:
        return f"{name} (All Params)"
    return f"{name} (Active: {pct:.2f}%)"

def generate_grouped_analysis(sub_df, scenario_title, all_probs_df, winner_type):
    """Generates the 3-part detailed impact plot (MLP, Heads, Heatmap)."""
    if len(sub_df) == 0:
        return

    # Prepare Data
    relevant_ids = sub_df['ID'].unique()
    current_probs = all_probs_df[all_probs_df['ID'].isin(relevant_ids)].copy()

    type_map = {'stereotype': 'Stereotype', 'anti-stereotype': 'Anti-Stereotype', 'unrelated': 'Unrelated'}
    current_probs['Type'] = current_probs['Type'].map(type_map).fillna(current_probs['Type'])
    plot_df = sub_df.copy()
    plot_df['Type'] = plot_df['Type'].map(type_map).fillna(plot_df['Type'])
    winner_label = type_map.get(winner_type, winner_type)

    # MLP Data
    mlp_df = plot_df[plot_df['Component'] == 'MLP']
    mlp_summary = mlp_df.groupby(['Layer', 'Type'])['Accumulated_Impact'].mean().reset_index()

    # Probability Bars Data
    prob_summary = current_probs.groupby(['Layer', 'Type'])['Layer_Accumulated_Prob'].mean().reset_index()
    prob_pivot = prob_summary.pivot(index='Layer', columns='Type', values='Layer_Accumulated_Prob').fillna(0)
    prob_summary_winner = pd.DataFrame({'Layer': prob_pivot.index})

    if winner_label == 'Stereotype':
        prob_summary_winner['Margin'] = prob_pivot.get('Stereotype', 0) - prob_pivot.get('Anti-Stereotype', 0)
        y_label = "Margin: P(Stereo) - P(Anti)"
    elif winner_label == 'Anti-Stereotype':
        prob_summary_winner['Margin'] = prob_pivot.get('Anti-Stereotype', 0) - prob_pivot.get('Stereotype', 0)
        y_label = "Margin: P(Anti) - P(Stereo)"
    else:
        prob_summary_winner['Margin'] = prob_pivot.get(winner_label, 0)
        y_label = f"Prob ({winner_label})"

    # Head Data & Heatmap Matrix
    head_df = plot_df[plot_df['Component'].str.startswith('Head')].copy()
    head_df['Head_ID'] = head_df['Component'].str.replace('Head_', '').astype(int)
    heatmap_data = head_df[head_df['Type'] == winner_label]
    head_matrix = heatmap_data.groupby(['Head_ID', 'Layer'])['Accumulated_Impact'].mean().unstack()

    head_layer_sum = head_df.groupby(['ID', 'Layer', 'Type'])['Accumulated_Impact'].sum().reset_index()
    head_sum_summary = head_layer_sum.groupby(['Layer', 'Type'])['Accumulated_Impact'].mean().reset_index()

    # Plotting Layout
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 0.9], hspace=0.3, wspace=0.25)

    ax_mlp = fig.add_subplot(gs[0, 0])
    ax_sum = fig.add_subplot(gs[0, 1])
    ax_heat = fig.add_subplot(gs[1, :])

    # --- PLOT 1: MLP Impact + Prob Bars ---
    ax_mlp_twin = ax_mlp.twinx()
    sns.barplot(
        data=prob_summary_winner, x='Layer', y='Margin',
        color='lightgray', alpha=0.5, ax=ax_mlp_twin, errorbar=None
    )
    ax_mlp_twin.set_ylabel(y_label, color='gray', fontweight='bold')
    ax_mlp_twin.grid(False)
    ax_mlp_twin.set_xticks([])

    sns.lineplot(
        data=mlp_summary, x='Layer', y='Accumulated_Impact', hue='Type',
        marker='o', linewidth=2.5, ax=ax_mlp, palette='colorblind'
    )
    ax_mlp.set_title("MLP Impact & Probability Margin", fontweight='bold')
    ax_mlp.set_ylabel("Cumulative Logit Contribution")
    ax_mlp.xaxis.set_major_locator(ticker.MultipleLocator(5))
    ax_mlp.legend(title="", loc='upper left', frameon=False)
    sns.despine(ax=ax_mlp, right=False)

    # --- PLOT 2: Head Sum ---
    sns.lineplot(
        data=head_sum_summary, x='Layer', y='Accumulated_Impact', hue='Type',
        marker='s', linewidth=2.5, ax=ax_sum, palette='colorblind'
    )
    ax_sum.set_title("Total Attention Block Impact (Sum of Heads)", fontweight='bold')
    ax_sum.set_ylabel("Cumulative Logit Contribution")
    ax_sum.xaxis.set_major_locator(ticker.MultipleLocator(5))
    ax_sum.legend(title="", loc='upper left', frameon=False)
    sns.despine(ax=ax_sum)

    # --- PLOT 3: Heatmap ---
    sns.heatmap(
        head_matrix, cmap="RdBu_r", center=0, ax=ax_heat,
        cbar_kws={'label': f'Impact on "{winner_label}" Logit', 'shrink': 0.8}
    )
    ax_heat.set_title(f"Detailed Head Impact Heatmap (Target: {winner_label})", fontweight='bold')
    ax_heat.set_xlabel("Layer Index")
    ax_heat.set_ylabel("Head Index")
    ax_heat.xaxis.set_major_locator(ticker.MultipleLocator(5))
    ax_heat.xaxis.set_major_formatter(ticker.ScalarFormatter())

    plt.suptitle(scenario_title, fontsize=18, fontweight='bold', y=0.96)

    # Add explanatory footnote
    plt.figtext(0.5, -0.02, FOOTNOTE_TEXT, ha='center', fontsize=11, style='italic', color='dimgray', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

    # Save safely
    safe_title = scenario_title.replace(" ", "_").replace("(", "").replace(")", "").replace(":", "").replace("%", "")
    s3_utils.save_plot(fig, f"outputs/gpt2-xl/.plots/{safe_title}_impact_analysis.pdf")
    plt.close(fig)

# =====================================================================
# 2. DATA PROCESSING & WINNER ASSIGNMENT
# =====================================================================
def processing(df_accumulated_impact, df_probability_info, fine_tuned=False, margin=0):
    if 'Layer_Accumulated_Prob' in df_accumulated_impact.columns:
        df_accumulated_impact = df_accumulated_impact.drop(columns=['Layer_Accumulated_Prob'])

    idx_max_tokens = df_probability_info.groupby(['ID', 'Type', 'Layer'])['Token_Position'].idxmax()
    prob_lookup = df_probability_info.loc[idx_max_tokens, ['ID', 'Type', 'Layer', 'Layer_Accumulated_Prob']].copy()

    df_accumulated_impact = pd.merge(
        df_accumulated_impact, prob_lookup, on=['ID', 'Type', 'Layer'], how='left'
    )

    max_layer = df_accumulated_impact['Layer'].max()
    last_token_indices = df_probability_info.groupby(['ID', 'Type'])['Token_Position'].transform('max') == df_probability_info['Token_Position']

    final_probs = df_probability_info[
        (df_probability_info['Layer'] == max_layer) & (last_token_indices)
    ].copy()

    if fine_tuned:
        final_probs = final_probs[final_probs['Type'].isin(['stereotype', 'anti-stereotype', 'unrelated'])]

    grouped_probs = final_probs.groupby(['ID', 'Type'])['Layer_Accumulated_Prob'].max().reset_index()
    prob_pivot = grouped_probs.pivot(index='ID', columns='Type', values='Layer_Accumulated_Prob').fillna(0)

    cols = prob_pivot.columns
    s_col = 'stereotype' if 'stereotype' in cols else None
    a_col = 'anti-stereotype' if 'anti-stereotype' in cols else None
    u_col = 'unrelated' if 'unrelated' in cols else None

    conditions, choices = [], []
    if s_col and a_col and u_col:
        conditions = [
            (prob_pivot[s_col] - prob_pivot[a_col] > margin) & (prob_pivot[s_col] - prob_pivot[u_col] > margin),
            (prob_pivot[a_col] - prob_pivot[s_col] > margin) & (prob_pivot[a_col] - prob_pivot[u_col] > margin),
            (prob_pivot[u_col] - prob_pivot[s_col] > margin) & (prob_pivot[u_col] - prob_pivot[a_col] > margin)
        ]

        # conditions = [
        #     (prob_pivot[s_col] - prob_pivot[a_col] > margin),
        #     (prob_pivot[a_col] - prob_pivot[s_col] >= margin)
        # ]
        choices = ['stereotype',
                   'anti-stereotype',
                   'unrelated'
                   ]
    elif s_col and a_col:
        conditions = [
            (prob_pivot[s_col] - prob_pivot[a_col] > margin),
            (prob_pivot[a_col] - prob_pivot[s_col] >= margin)
        ]
        choices = ['stereotype', 'anti-stereotype']

    if conditions:
        prob_pivot['Winner_Type'] = np.select(conditions, choices, default='neutral')
    else:
        prob_pivot['Winner_Type'] = 'unknown'

    id_to_winner = prob_pivot['Winner_Type'].to_dict()

    df_accumulated_impact['Model_Preference'] = df_accumulated_impact['ID'].map(id_to_winner)
    df_probability_info['Model_Preference'] = df_probability_info['ID'].map(id_to_winner)

    return df_accumulated_impact, df_probability_info

# =====================================================================
# 3. MULTI-MODEL ANALYSIS & PLOTTING
# =====================================================================
def analyze_multiple_models(models_dict, bias_type="gender"):

    # Map Bias Types
    try:
        raw_data = s3_utils.read_json("data/stereoset/test.json")
        full_intrasentence_list = raw_data.get('data', {}).get('intrasentence', [])
        id_to_biastype = {item['id']: item['bias_type'] for item in full_intrasentence_list}
        del raw_data; gc.collect()
    except Exception as e:
        print(f"Warning: Could not load JSON from S3. Mapping all IDs to '{bias_type}'.")
        id_to_biastype = {i: bias_type for i in range(10000)}

    processed_impacts = {}
    processed_probs = {}

    # Format model names with their trainable parameter percentages
    formatted_models_dict = {format_model_name(k): v for k, v in models_dict.items()}

    # Process Data
    for model_name, (df_prob, df_impact) in formatted_models_dict.items():
        df_prob['Bias_Type'] = df_prob['ID'].map(id_to_biastype).fillna('unknown')
        df_impact['Bias_Type'] = df_impact['ID'].map(id_to_biastype).fillna('unknown')

        is_ft = "Original" not in model_name
        p_impact, p_prob = processing(df_impact, df_prob, fine_tuned=is_ft)

        processed_impacts[model_name] = p_impact[p_impact['Bias_Type'] == bias_type].copy()
        processed_probs[model_name] = p_prob[p_prob['Bias_Type'] == bias_type].copy()

    if not any(len(df) > 0 for df in processed_impacts.values()):
        print(f"No data found for bias type '{bias_type}'.")
        return processed_impacts, processed_probs

    # -----------------------------------------------------------------
    # PLOT A: Global Counts
    # -----------------------------------------------------------------
    count_dfs = []
    reverse_name_map = {format_model_name(k): k for k in models_dict.keys()}

    for model_name, p_impact in processed_impacts.items():
        counts = p_impact[['ID', 'Model_Preference']].drop_duplicates()['Model_Preference'].value_counts().reset_index()
        counts.columns = ['Preference_Type', 'Count']
        counts['Model'] = model_name

        base_name = reverse_name_map[model_name]
        counts['Active_Percentage'] = PARAM_PERCENTAGES.get(base_name, 100.0)

        # Categorize the fine-tuning method for trendlines
        if "Attn Impact MLP" in base_name:
            counts['Method'] = "Attn Impact MLP"
        elif "Attn" in base_name:
            counts['Method'] = "Attn Only"
        else:
            counts['Method'] = "Original Baseline"

        count_dfs.append(counts)

    comparison_counts = pd.concat(count_dfs, ignore_index=True)
    comparison_counts = comparison_counts.sort_values('Active_Percentage')

    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=comparison_counts, x='Preference_Type', y='Count',
        hue='Model', palette='viridis'
    )
    plt.title(f"Global Preference Counts Across Configurations ({bias_type.capitalize()})", fontweight='bold')
    plt.ylabel("Number of Examples")
    plt.xlabel("Model Output Preference")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
    sns.despine()

    plt.figtext(0.5, -0.05, FOOTNOTE_TEXT, ha='center', fontsize=11, style='italic', color='dimgray', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
    s3_utils.save_plot(plt.gcf(), "outputs/gpt2-xl/.plots/global_preference_counts.pdf")
    plt.close()

    # -----------------------------------------------------------------
    # PLOT A2: Preference Counts vs Active Parameters
    # -----------------------------------------------------------------
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        data=comparison_counts,
        x='Active_Percentage',
        y='Count',
        hue='Preference_Type',
        style='Method',
        markers=True,
        dashes=False,
        palette='viridis',
        linewidth=2,
        markersize=8
    )
    plt.title(f"Preference Counts vs Active Parameters ({bias_type.capitalize()})", fontweight='bold')
    plt.ylabel("Number of Examples")
    plt.xlabel("Active Parameters (%) - Log Scale")
    plt.xscale('log')

    # Format X-axis to display regular scalar numbers instead of scientific notation
    ax = plt.gca()
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: '{:g}'.format(y)))

    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
    sns.despine()

    plt.figtext(0.5, -0.05, FOOTNOTE_TEXT, ha='center', fontsize=11, style='italic', color='dimgray', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
    s3_utils.save_plot(plt.gcf(), "outputs/gpt2-xl/.plots/global_preference_vs_parameters.pdf")
    plt.close()

    # -----------------------------------------------------------------
    # PLOT B: Overall Layerwise Probability Comparison
    # -----------------------------------------------------------------
    prefs_to_plot = ['stereotype', 'anti-stereotype', 'unrelated']
    all_probs_list = []
    for model_name, p_prob in processed_probs.items():
        clean_probs = p_prob[['ID', 'Layer', 'Type', 'Layer_Accumulated_Prob']].drop_duplicates().copy()
        clean_probs['Model'] = model_name
        all_probs_list.append(clean_probs)

    combined_probs = pd.concat(all_probs_list, ignore_index=True)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    fig.suptitle(f"Layerwise Probability by Type Across Configurations ({bias_type.capitalize()})", fontweight='bold', fontsize=16, y=1.05)

    for ax, t in zip(axes, prefs_to_plot):
        subset = combined_probs[combined_probs['Type'] == t]
        if len(subset) == 0:
            ax.axis('off')
            continue

        sns.lineplot(
            data=subset, x='Layer', y='Layer_Accumulated_Prob', hue='Model',
            ax=ax, linewidth=2, palette='viridis'
        )
        ax.set_title(f"Mean {t.capitalize()} Prob", fontweight='bold')
        ax.set_xlabel("Layer")
        if ax == axes[0]:
            ax.set_ylabel("Accumulated Output Probability")

        # Only keep legend on the last plot
        if ax == axes[-1]:
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
        else:
            ax.get_legend().remove()

    sns.despine()
    plt.figtext(0.5, -0.05, FOOTNOTE_TEXT, ha='center', fontsize=11, style='italic', color='dimgray', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
    s3_utils.save_plot(fig, "outputs/gpt2-xl/.plots/layerwise_probs_comparison.pdf")
    plt.close(fig)

    # -----------------------------------------------------------------
    # PLOT C: Distribution of Margins (Stereotype vs Anti-Stereotype)
    # -----------------------------------------------------------------
    margin_dfs = []
    for model_name, p_prob in processed_probs.items():
        max_layer = p_prob['Layer'].max()
        final_layer_probs = p_prob[p_prob['Layer'] == max_layer].groupby(['ID', 'Type'])['Layer_Accumulated_Prob'].max().reset_index()
        pivot_probs = final_layer_probs.pivot(index='ID', columns='Type', values='Layer_Accumulated_Prob').fillna(0)

        if 'stereotype' in pivot_probs.columns and 'anti-stereotype' in pivot_probs.columns:
            filtered_probs = pivot_probs[pivot_probs['stereotype'] > pivot_probs['anti-stereotype']]

            if not filtered_probs.empty:
                abs_margin = np.abs(filtered_probs['stereotype'] - filtered_probs['anti-stereotype'])
                log10_margin = np.log10(np.clip(abs_margin, a_min=1e-20, a_max=None))

                margins = pd.DataFrame({
                    'ID': filtered_probs.index,
                    'Margin': abs_margin,
                    'Log10_Margin': log10_margin,
                    'Model': model_name
                })
                margin_dfs.append(margins)

    if margin_dfs:
        combined_margins = pd.concat(margin_dfs, ignore_index=True)

        plt.figure(figsize=(12, 6))
        sns.histplot(
            data=combined_margins, x='Log10_Margin', hue='Model',
            kde=True, element='step', stat='density',
            common_norm=False, palette='viridis', alpha=0.3, linewidth=1.5
        )
        plt.title(rf"Density of $\log_{{10}}(\Delta P)$ [Stereo - Anti] at Final Layer ({bias_type.capitalize()})", fontweight='bold')
        plt.xlabel(r"$\log_{10}$ (Probability Margin)")
        plt.ylabel("Density")

        # Move legend outside for clean look
        sns.move_legend(plt.gca(), "upper left", bbox_to_anchor=(1, 1), frameon=False)
        sns.despine()
        plt.figtext(0.5, -0.05, FOOTNOTE_TEXT, ha='center', fontsize=11, style='italic', color='dimgray', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
        s3_utils.save_plot(plt.gcf(), "outputs/gpt2-xl/.plots/margin_distribution.pdf")
        plt.close()

    # -----------------------------------------------------------------
    # PLOT D: Stereotype Score (SS) per Model
    # -----------------------------------------------------------------
    ss_records = []
    for model_name, p_prob in processed_probs.items():
        max_layer = p_prob['Layer'].max()
        last_tok_idx = p_prob.groupby(['ID', 'Type', 'Layer'])['Token_Position'].idxmax()
        final = p_prob.loc[last_tok_idx]
        final = final[final['Layer'] == max_layer]
        pivot = final.pivot(index='ID', columns='Type', values='Layer_Accumulated_Prob').fillna(0)

        n_stereo = 0
        n_anti = 0
        if 'stereotype' in pivot.columns and 'anti-stereotype' in pivot.columns:
            n_stereo = int((pivot['stereotype'] > pivot['anti-stereotype']).sum())
            n_anti = int((pivot['anti-stereotype'] > pivot['stereotype']).sum())

        denom = n_stereo + n_anti
        ss = (n_stereo / denom * 100) if denom > 0 else 50.0

        base_name = reverse_name_map.get(model_name, model_name)
        ss_records.append({
            'Model': model_name,
            'SS': ss,
            'Active_Percentage': PARAM_PERCENTAGES.get(base_name, 100.0),
            'N_stereo': n_stereo,
            'N_anti': n_anti,
        })

    ss_df = pd.DataFrame(ss_records).sort_values('Active_Percentage')

    fig, ax = plt.subplots(figsize=(max(8, len(ss_df) * 1.5), 6))
    bars = ax.bar(range(len(ss_df)), ss_df['SS'], color=sns.color_palette('viridis', len(ss_df)))
    ax.axhline(y=50, color='red', linestyle='--', linewidth=1.5, label='Ideal (SS = 50)')
    ax.set_xticks(range(len(ss_df)))
    ax.set_xticklabels(ss_df['Model'], rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('Stereotype Score (SS)')
    ax.set_ylim(0, 100)
    ax.set_title(f'Stereotype Score Across Configurations ({bias_type.capitalize()})', fontweight='bold')
    ax.legend(frameon=False)

    for i, (_, row) in enumerate(ss_df.iterrows()):
        ax.text(i, row['SS'] + 1.5, f"{row['SS']:.1f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

    sns.despine()
    plt.figtext(0.5, -0.05, FOOTNOTE_TEXT, ha='center', fontsize=11, style='italic', color='dimgray',
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
    s3_utils.save_plot(fig, "outputs/gpt2-xl/.plots/stereotype_score_comparison.pdf")
    plt.close(fig)

    # -----------------------------------------------------------------
    # PLOT E: LMS and ICAT Scores per Model
    # -----------------------------------------------------------------
    metric_records = []

    for model_name, p_prob in processed_probs.items():
        max_layer = p_prob['Layer'].max()
        last_tok_idx = p_prob.groupby(['ID', 'Type', 'Layer'])['Token_Position'].idxmax()
        final = p_prob.loc[last_tok_idx]
        final = final[final['Layer'] == max_layer]
        pivot = final.pivot(index='ID', columns='Type', values='Layer_Accumulated_Prob').fillna(0)

        n_total = len(pivot)
        related = 0
        n_stereo = 0
        n_anti = 0

        has_all_three = {'stereotype', 'anti-stereotype', 'unrelated'} <= set(pivot.columns)
        has_stereo_anti = {'stereotype', 'anti-stereotype'} <= set(pivot.columns)

        if has_all_three:
            related += int((pivot['stereotype'] > pivot['unrelated']).sum())
            related += int((pivot['anti-stereotype'] > pivot['unrelated']).sum())
        if has_stereo_anti:
            n_stereo = int((pivot['stereotype'] > pivot['anti-stereotype']).sum())
            n_anti = int((pivot['anti-stereotype'] > pivot['stereotype']).sum())

        lms = (related / (2 * n_total) * 100) if (n_total > 0 and has_all_three) else 0.0
        denom = n_stereo + n_anti
        ss = (n_stereo / denom * 100) if denom > 0 else 50.0
        icat = lms * (min(ss, 100.0 - ss) / 50.0)

        base_name = reverse_name_map.get(model_name, model_name)
        metric_records.append({
            'Model': model_name,
            'LMS': lms,
            'SS': ss,
            'ICAT': icat,
            'Active_Percentage': PARAM_PERCENTAGES.get(base_name, 100.0),
        })

    if metric_records:
        metric_df = pd.DataFrame(metric_records).sort_values('Active_Percentage')

        fig, (ax_lms, ax_icat) = plt.subplots(1, 2, figsize=(max(14, len(metric_df) * 2.5), 6))

        bar_colors = sns.color_palette('viridis', len(metric_df))

        ax_lms.bar(range(len(metric_df)), metric_df['LMS'], color=bar_colors)
        ax_lms.axhline(y=100, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='Perfect LMS (100)')
        ax_lms.set_xticks(range(len(metric_df)))
        ax_lms.set_xticklabels(metric_df['Model'], rotation=35, ha='right', fontsize=8)
        ax_lms.set_ylabel('LMS (%)')
        ax_lms.set_ylim(0, 110)
        ax_lms.set_title('Language Modeling Score (LMS)', fontweight='bold')
        ax_lms.legend(frameon=False, fontsize=9)
        for i, (_, row) in enumerate(metric_df.iterrows()):
            ax_lms.text(i, row['LMS'] + 1.5, f"{row['LMS']:.1f}", ha='center', va='bottom', fontsize=9, fontweight='bold')
        sns.despine(ax=ax_lms)

        ax_icat.bar(range(len(metric_df)), metric_df['ICAT'], color=bar_colors)
        ax_icat.axhline(y=100, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='Perfect ICAT (100)')
        ax_icat.set_xticks(range(len(metric_df)))
        ax_icat.set_xticklabels(metric_df['Model'], rotation=35, ha='right', fontsize=8)
        ax_icat.set_ylabel('ICAT')
        ax_icat.set_ylim(0, max(110, metric_df['ICAT'].max() + 10))
        ax_icat.set_title('Idealized CAT (ICAT)', fontweight='bold')
        ax_icat.legend(frameon=False, fontsize=9)
        for i, (_, row) in enumerate(metric_df.iterrows()):
            ax_icat.text(i, row['ICAT'] + 1.5, f"{row['ICAT']:.1f}", ha='center', va='bottom', fontsize=9, fontweight='bold')
        sns.despine(ax=ax_icat)

        fig.suptitle(f'LMS & ICAT Across Configurations ({bias_type.capitalize()})',
                     fontweight='bold', fontsize=14, y=1.02)
        plt.figtext(0.5, -0.05, FOOTNOTE_TEXT, ha='center', fontsize=11, style='italic', color='dimgray',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
        s3_utils.save_plot(fig, "outputs/gpt2-xl/.plots/lms_icat_comparison.pdf")
        plt.close(fig)

    return processed_impacts, processed_probs

In [4]:
df_probability_info = s3_utils.read_csv("outputs/gpt2-xl/dev_tests/out_DLA_gender_test.csv")
df_accumulated_impact = s3_utils.read_csv("outputs/gpt2-xl/dev_tests/accumulated_impact_gender_test.csv")

# df_probability_info_gender_ft_0_2 = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_0.2.csv")
# df_accumulated_impact_gender_ft_0_2 = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_0.2.csv")

# df_probability_info_gender_ft_0_5 = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_0.5.csv")
# df_accumulated_impact_gender_ft_0_5 = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_0.5.csv")

# df_probability_info_gender_ft_0_8 = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_0.8.csv")
# df_accumulated_impact_gender_ft_0_8 = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_0.8.csv")

# df_probability_info_gender_ft_1 = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_1.csv")
# df_accumulated_impact_gender_ft_1 = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_1.csv")

# df_probability_info_gender_ft_5 = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_5.csv")
# df_accumulated_impact_gender_ft_5 = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_5.csv")

# df_probability_info_gender_ft_10 = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_10.csv")
# df_accumulated_impact_gender_ft_10 = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_10.csv")

# df_probability_info_gender_ft_20 = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_20.csv")
# df_accumulated_impact_gender_ft_20 = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_20.csv")

# # df_probability_info_gender_ft_1_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_mlp_impact_only_1.csv")
# # df_accumulated_impact_gender_ft_1_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_mlp_impact_only_2.csv")
# #
# # df_probability_info_gender_ft_5_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_mlp_impact_only_5.csv")
# # df_accumulated_impact_gender_ft_5_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_mlp_impact_only_5.csv")
# #
# # df_probability_info_gender_ft_10_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_mlp_impact_only_10.csv")
# # df_accumulated_impact_gender_ft_10_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_mlp_impact_only_10.csv")
# #
# # df_probability_info_gender_ft_20_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_mlp_impact_only_20.csv")
# # df_accumulated_impact_gender_ft_20_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_mlp_impact_only_20.csv")


# df_probability_info_gender_ft_0_2_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_impact_mlp_0.2.csv")
# df_accumulated_impact_gender_ft_0_2_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_impact_mlp_0.2.csv")

# df_probability_info_gender_ft_0_5_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_impact_mlp_0.5.csv")
# df_accumulated_impact_gender_ft_0_5_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_impact_mlp_0.5.csv")

# df_probability_info_gender_ft_0_8_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_impact_mlp_0.8.csv")
# df_accumulated_impact_gender_ft_0_8_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_impact_mlp_0.8.csv")

# df_probability_info_gender_ft_1_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_impact_mlp_1.csv")
# df_accumulated_impact_gender_ft_1_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_impact_mlp_1.csv")

# df_probability_info_gender_ft_2_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_impact_mlp_2.csv")
# df_accumulated_impact_gender_ft_2_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_impact_mlp_2.csv")

# df_probability_info_gender_ft_5_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_impact_mlp_5.csv")
# df_accumulated_impact_gender_ft_5_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_impact_mlp_5.csv")

# df_probability_info_gender_ft_10_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_impact_mlp_10.csv")
# df_accumulated_impact_gender_ft_10_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_impact_mlp_10.csv")

# df_probability_info_gender_ft_20_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_attn_impact_mlp_20.csv")
# df_accumulated_impact_gender_ft_20_attn_impact_mlp = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_attn_impact_mlp_20.csv")


# df_probability_info_gender_ft_100_full = pd.read_csv("outputs/gpt2-xl/fine_tuned/out_DLA_gender_test_fine_tuned_full_100.csv")
# df_accumulated_impact_gender_ft_100_full = pd.read_csv("outputs/gpt2-xl/fine_tuned/accumulated_impact_gender_test_fine_tuned_full_100.csv")


# =========== DPO ===========
# df_probability_info_gender_ft_0_2 = pd.read_csv("outputs/gpt2-xl/fine_tuned/dpo/out_DLA_gender_test_fine_tuned_attn_0.2.csv")
# df_accumulated_impact_gender_ft_0_2 = pd.read_csv("outputs/gpt2-xl/fine_tuned/dpo/accumulated_impact_gender_test_fine_tuned_attn_0.2.csv")

# df_probability_info_gender_ft_0_5 = pd.read_csv("outputs/gpt2-xl/fine_tuned/dpo/out_DLA_gender_test_fine_tuned_attn_0.5.csv")
# df_accumulated_impact_gender_ft_0_5 = pd.read_csv("outputs/gpt2-xl/fine_tuned/dpo/accumulated_impact_gender_test_fine_tuned_attn_0.5.csv")

# df_probability_info_gender_ft_0_8 = pd.read_csv("outputs/gpt2-xl/fine_tuned/dpo/out_DLA_gender_test_fine_tuned_attn_0.8.csv")
# df_accumulated_impact_gender_ft_0_8 = pd.read_csv("outputs/gpt2-xl/fine_tuned/dpo/accumulated_impact_gender_test_fine_tuned_attn_0.8.csv")

# df_probability_info_gender_ft_1 = pd.read_csv("outputs/gpt2-xl/fine_tuned/dpo/out_DLA_gender_test_fine_tuned_attn_1.0.csv")
# df_accumulated_impact_gender_ft_1 = pd.read_csv("outputs/gpt2-xl/fine_tuned/dpo/accumulated_impact_gender_test_fine_tuned_attn_1.0.csv")

df_probability_info_gender_ft_5_attn_impact_mlp = s3_utils.read_csv("outputs/gpt2-xl/fine_tuned/dpo/out_DLA_gender_test_fine_tuned_attn_impact_mlp_5.0.csv")
df_accumulated_impact_gender_ft_5_attn_impact_mlp = s3_utils.read_csv("outputs/gpt2-xl/fine_tuned/dpo/accumulated_impact_gender_test_fine_tuned_attn_impact_mlp_5.0.csv")

df_probability_info = df_probability_info[df_probability_info['ID'].isin(df_probability_info_gender_ft_5_attn_impact_mlp['ID'])]
df_accumulated_impact = df_accumulated_impact[df_accumulated_impact['ID'].isin(df_accumulated_impact_gender_ft_5_attn_impact_mlp['ID'])]


models_data = {
    "Original": (df_probability_info, df_accumulated_impact),
    # "Fine-Tuned (Attn 0.2)": (df_probability_info_gender_ft_0_2, df_accumulated_impact_gender_ft_0_2),
    # "Fine-Tuned (Attn 0.5)": (df_probability_info_gender_ft_0_5, df_accumulated_impact_gender_ft_0_5), # <-- Make sure you use 0.5 vars here
    # "Fine-Tuned (Attn 0.8)": (df_probability_info_gender_ft_0_8, df_accumulated_impact_gender_ft_0_8), # <-- Make sure you use 0.8 vars here
    # "Fine-Tuned (Attn 1)": (df_probability_info_gender_ft_1, df_accumulated_impact_gender_ft_1),
    # "Fine-Tuned (Attn 5)": (df_probability_info_gender_ft_5, df_accumulated_impact_gender_ft_5),
    # "Fine-Tuned (Attn 10)": (df_probability_info_gender_ft_10, df_accumulated_impact_gender_ft_10),
    # "Fine-Tuned (Attn 20)": (df_probability_info_gender_ft_20, df_accumulated_impact_gender_ft_20),

    # "Fine-Tuned (Attn Impact MLP 0.2)" : (df_probability_info_gender_ft_0_2_attn_impact_mlp, df_accumulated_impact_gender_ft_0_2_attn_impact_mlp),
    # "Fine-Tuned (Attn Impact MLP 0.5)" : (df_probability_info_gender_ft_0_5_attn_impact_mlp, df_accumulated_impact_gender_ft_0_5_attn_impact_mlp),
    # "Fine-Tuned (Attn Impact MLP 0.8)" : (df_probability_info_gender_ft_0_8_attn_impact_mlp, df_accumulated_impact_gender_ft_0_8_attn_impact_mlp),
    # "Fine-Tuned (Attn Impact MLP 1)" : (df_probability_info_gender_ft_1_attn_impact_mlp, df_accumulated_impact_gender_ft_1_attn_impact_mlp),
    # "Fine-Tuned (Attn Impact MLP 2)" : (df_probability_info_gender_ft_2_attn_impact_mlp, df_accumulated_impact_gender_ft_2_attn_impact_mlp),
    "Fine-Tuned (Attn Impact MLP 5)" : (df_probability_info_gender_ft_5_attn_impact_mlp, df_accumulated_impact_gender_ft_5_attn_impact_mlp),
    # "Fine-Tuned (Attn Impact MLP 10)" : (df_probability_info_gender_ft_10_attn_impact_mlp, df_accumulated_impact_gender_ft_10_attn_impact_mlp),
    # "Fine-Tuned (Attn Impact MLP 20)" : (df_probability_info_gender_ft_20_attn_impact_mlp, df_accumulated_impact_gender_ft_20_attn_impact_mlp),

    # "Fine-Tuned (Full 100)" : (df_probability_info_gender_ft_100_full, df_accumulated_impact_gender_ft_100_full),
}

In [5]:
bias_type = "gender"

processed_impacts, processed_probs = analyze_multiple_models(models_dict=models_data, bias_type=bias_type)

In [ ]:
for base_model_name in models_data.keys():
    # Retrieve formatted name
    formatted_name = format_model_name(base_model_name)
    print(f"--- Generating detailed impact .plots for: {formatted_name} ---")

    for pref in ['stereotype', 'anti-stereotype', 'unrelated']:
        model_impact_df = processed_impacts[formatted_name]
        model_prob_df = processed_probs[formatted_name]

        df_sub = model_impact_df[model_impact_df['Model_Preference'] == pref]

        if len(df_sub) > 0:
            generate_grouped_analysis(
                sub_df=df_sub,
                scenario_title=f"[{bias_type.capitalize()}] {formatted_name}\nPreferred: {pref.capitalize()}",
                all_probs_df=model_prob_df,
                winner_type=pref
            )